In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [ ]:
df = pd.read_csv("Reviews.csv", nrows=60000)

df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Id                      60000 non-null  int64 
 1   ProductId               60000 non-null  object
 2   UserId                  60000 non-null  object
 3   ProfileName             59995 non-null  object
 4   HelpfulnessNumerator    60000 non-null  int64 
 5   HelpfulnessDenominator  60000 non-null  int64 
 6   Score                   60000 non-null  int64 
 7   Time                    60000 non-null  int64 
 8   Summary                 59998 non-null  object
 9   Text                    60000 non-null  object
dtypes: int64(5), object(5)
memory usage: 4.6+ MB


In [ ]:
df.isnull().sum()

,0
Id,0
ProductId,0
UserId,0
ProfileName,5
HelpfulnessNumerator,0
HelpfulnessDenominator,0
Score,0
Time,0
Summary,2
Text,0


In [ ]:
df.describe()

,Id,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time
count,60000.000000,60000.000000,60000.000000,60000.000000,6.000000e+04
mean,30000.500000,1.623717,2.095583,4.151333,1.295236e+09
std,17320.652413,5.448381,6.106287,1.329384,4.734717e+07
min,1.000000,0.000000,0.000000,1.000000,9.617184e+08
25%,15000.750000,0.000000,0.000000,4.000000,1.270080e+09
50%,30000.500000,0.000000,1.000000,5.000000,1.308960e+09
75%,45000.250000,2.000000,2.000000,5.000000,1.331165e+09
max,60000.000000,398.000000,401.000000,5.000000,1.351210e+09


In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df["Score"].fillna(df["Score"].median(), inplace=True)

/tmp/ipykernel_2999/1700578021.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Score"].fillna(df["Score"].median(), inplace=True)


In [ ]:
features = [

"Score",

"HelpfulnessNumerator",

"HelpfulnessDenominator",

"ReviewLength",

"SummaryLength",

"HelpfulRatio",

"ReviewYear"

]

In [ ]:
target = "Score"

In [ ]:
df["ReviewYear"] = pd.to_datetime(df["Time"], unit="s").dt.year

In [ ]:
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,ReviewLength,SummaryLength,HelpfulRatio,ReviewYear
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...,263,21.0,0.50,2011
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...,190,17.0,0.00,2012
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...,509,21.0,0.50,2008
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...,219,14.0,0.75,2011
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...,140,11.0,0.00,2012


In [ ]:
df.isnull().sum()

,0
Id,0
ProductId,0
UserId,0
ProfileName,5
HelpfulnessNumerator,0
HelpfulnessDenominator,0
Score,0
Time,0
Summary,2
Text,0


In [ ]:
df["ProfileName"].fillna("Unknown", inplace=True)

df["Summary"].fillna("No Summary", inplace=True)

df["Text"].fillna("No Review", inplace=True)

/tmp/ipykernel_2999/2488491788.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["ProfileName"].fillna("Unknown", inplace=True)


In [ ]:
df["SummaryLength"] = df["Summary"].apply(len)

In [ ]:
df["HelpfulRatio"] = (
    df["HelpfulnessNumerator"] /
    (df["HelpfulnessDenominator"] + 1)
)

In [ ]:
df["ReviewYear"] = pd.to_datetime(
    df["Time"],
    unit="s"
).dt.year

In [ ]:
features = [
    "HelpfulnessNumerator",
    "HelpfulnessDenominator",
    "ReviewLength",
    "SummaryLength",
    "HelpfulRatio",
    "ReviewYear"
]

target = "Score"

In [ ]:
X = df[features]

y = df[target]

In [ ]:
X.head()

,HelpfulnessNumerator,HelpfulnessDenominator,ReviewLength,SummaryLength,HelpfulRatio,ReviewYear
0,1,1,263,21,0.50,2011
1,0,0,190,17,0.00,2012
2,1,1,509,21,0.50,2008
3,3,3,219,14,0.75,2011
4,0,0,140,11,0.00,2012


In [ ]:
num_cols = [
    "HelpfulnessNumerator",
    "HelpfulnessDenominator",
    "ReviewLength",
    "SummaryLength",
    "HelpfulRatio",
    "ReviewYear"
]

cat_cols = []

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer


numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_cols)
])

In [ ]:
from sklearn.model_selection import train_test_split


X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)


X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

In [ ]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(42000, 6)
(9000, 6)
(9000, 6)


In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)

X_test_processed = preprocessor.transform(X_test)

In [ ]:
import joblib

joblib.dump(
    preprocessor,
    "Amazon_Food_Preprocessing_Pipeline.pkl"
)

print("Pipeline Saved Successfully")

Pipeline Saved Successfully


In [ ]:
class AmazonPipeline:

    def __init__(self):
        self.preprocessor = preprocessor

    def fit(self, X):
        self.preprocessor.fit(X)

    def transform(self, X):
        return self.preprocessor.transform(X)

    def save(self, filename):
        joblib.dump(self.preprocessor, filename)

In [ ]:
amazon_pipeline = AmazonPipeline()

amazon_pipeline.fit(X_train)

amazon_pipeline.save("AmazonPipeline.pkl")

print("Amazon Pipeline Saved Successfully")

Amazon Pipeline Saved Successfully


In [ ]:
import pandas as pd

processed_train = pd.DataFrame(
    X_train_processed,
    columns=num_cols
)

processed_train.to_csv(
    "processed_amazon_reviews.csv",
    index=False
)

print("Processed Dataset Saved Successfully")

Processed Dataset Saved Successfully


In [ ]:
processed_train.head()

,HelpfulnessNumerator,HelpfulnessDenominator,ReviewLength,SummaryLength,HelpfulRatio,ReviewYear
0,-0.284114,-0.329376,-0.534346,-0.816953,-0.853523,0.954476
1,-0.284114,-0.329376,-0.168542,-0.385604,-0.853523,-2.366156
2,-0.284114,0.295838,-0.813389,-0.888844,-0.853523,-1.037903
3,-0.110410,-0.173073,-0.445240,-0.816953,0.730247,-1.037903
4,-0.110410,-0.173073,-0.801665,-1.032627,0.730247,-1.037903
